# GPU 활용 임베딩 생성 (Google Colab)

이 노트북은 `generate_embeddings_v3_parallel.py`를 사용하여 가속화된 임베딩 생성을 수행합니다.
Colab의 로컬 디스크 용량 한계를 고려하여 최적화된 설정을 사용합니다.

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 필수 패키지 설치

In [ ]:
!pip install duckdb pandas pyarrow torch tqdm

## 3. 프로젝트 코드 준비
구글 드라이브에 있는 프로젝트 폴더로 이동하거나 코드를 복사해 옵니다.

In [ ]:
# GitHub 저장소 클론 (프로젝트 코드)
import os
from google.colab import userdata

# 🔑 아이콘 클릭 → Add new secret
# Name: GITHUB_TOKEN
# Value: your_personal_access_token
try:
    token = userdata.get('GITHUB_TOKEN')
    use_token = True
    print("🔑 Using GitHub token for authentication")
except:
    print("⚠️ GITHUB_TOKEN not found. Using public clone (may have rate limits)")
    use_token = False

repo_path = '/content/stock-bot2'

# 이미 클론되어 있으면 스킵
if not os.path.exists(repo_path):
    print("📥 Cloning repository...")
    if use_token:
        !git clone https://{token}@github.com/gblue1223/stock-bot2.git {repo_path}
    else:
        !git clone https://github.com/gblue1223/stock-bot2.git {repo_path}

    if os.path.exists(repo_path):
        print("✅ Repository cloned successfully!")
    else:
        print("❌ Repository cloning failed. Please check your GitHub token and repository access permissions.")
        # Early exit if cloning failed to prevent further errors
        # No need to change directory or sys path if clone failed
        print(f"📂 Current directory: {os.getcwd()}")
        sys.exit("Repository cloning failed.") # Stop execution here.

else:
    print("📁 Repository already exists, updating...")
    %cd {repo_path}
    !git fetch origin && git pull
    print("✅ Repository updated!")

# 작업 디렉토리 변경 및 Python path 추가 (성공적으로 클론되거나 업데이트된 경우에만)
%cd {repo_path}
import sys
# Ensure the project root is in the Python path
if repo_path not in sys.path:
    sys.path.append(repo_path)

print("✅ Repository ready!")
print(f"📂 Current directory: {os.getcwd()}")

PROJECT_PATH = '/content/stock-bot2'
%cd {PROJECT_PATH}

## 4. 데이터 로컬 복사 (속도 최적화)
구글 드라이브에서 직접 DB를 읽으면 IO 속도가 매우 느려 병렬 처리가 어렵습니다.
DB 파일을 Colab 로컬 저장소(`/content`)로 복사합니다.

In [ ]:
# 드라이브의 원본 DB 경로
DRIVE_DB_PATH = '/content/drive/MyDrive/ColabData/datasets/stockbot/20260117/datasets_raw_09_11.duckdb'
LOCAL_DB_PATH = '/content/datasets.duckdb'

if not os.path.exists(LOCAL_DB_PATH):
    print("DB 파일 복사 중...")
    !cp {DRIVE_DB_PATH} {LOCAL_DB_PATH}
    print("복사 완료.")

## 5. 임베딩 생성 실행
경로 설정 시 `--output_dir`과 `--state_file`은 구글 드라이브 경로를 지정하여 중간 진행 상황이 저장되도록 합니다.

**주의:** `generate_embeddings_v3_parallel.py` 내의 `MAX_TEMP_SIZE`가 500GB로 설정되어 있다면, Colab 환경에 맞춰 **30GB** 정도로 수정하는 것을 권장합니다.

In [ ]:
!export MAX_TEMP_SIZE=50
!python scripts/data/generate_embeddings_colab.py \
  --db_path "/content/datasets.duckdb" \
  --model_path "/content/drive/MyDrive/ColabData/datasets/stockbot/20260117/pre_training_data/best_model.pt" \
  --output_dir "/content/drive/MyDrive/ColabData/datasets/stockbot/20260117/embeddings_v2" \
  --state_file "/content/drive/MyDrive/ColabData/datasets/stockbot/20260117/embeddings_v2/processed_stocks.txt" \
  --table_name "datasets" \
  --seq_len 120 \
  --batch_size 4096 \
  --device cuda

# 검증
Execute a single consolidated Python script to robustly verify and recover stock embeddings. The script must:
1.  **Setup**: Force remount Google Drive, ensure the project repository exists at `"/content/stock-bot2"`, and verify the local database `"/content/datasets.duckdb"` is ready (copying from `"/content/drive/MyDrive/ColabData/datasets/stockbot/20260117/datasets_raw_09_11.duckdb"` if necessary).
2.  **Verify**: Perform chunked verification of parquet files in `"/content/drive/MyDrive/ColabData/datasets/stockbot/20260117/embeddings_v2"` by copying them locally to `"/content/temp_verify"`, utilizing a state file to resume from interruptions.
3.  **Recover**: Identify missing stock embeddings, create a temporary retry database `"/content/datasets_retry.duckdb"`, and execute the `generate_embeddings_colab.py` script using the model at `"/content/drive/MyDrive/ColabData/datasets/stockbot/20260117/pre_training_data/best_model.pt"` to regenerate them.
4.  **Finalize**: Perform a final check of the generated files and clean up all temporary directories and state files.

## Consolidated Robust Verification and Recovery

### Subtask:
Generate and execute a single, comprehensive Python script to verify all embeddings, recover missing ones, and perform final cleanup.


In [ ]:
import duckdb
import os
import glob
import shutil
import math
import sys
import time
import subprocess
from tqdm import tqdm
from IPython import get_ipython
from google.colab import drive
from google.colab import userdata

# --- Configuration ---
PROJECT_PATH = '/content/stock-bot2'
DB_PATH = '/content/datasets.duckdb'
DRIVE_DB_PATH = '/content/drive/MyDrive/ColabData/datasets/stockbot/20260117/datasets_raw_09_11.duckdb'
OUTPUT_DIR = "/content/drive/MyDrive/ColabData/datasets/stockbot/20260117/embeddings_v2"
MODEL_PATH = "/content/drive/MyDrive/ColabData/datasets/stockbot/20260117/pre_training_data/best_model.pt"

# Temp Files
LOCAL_TEMP_VERIFY = "/content/temp_verify_robust"
VERIFICATION_STATE_FILE = "/content/verification_state.txt"
RETRY_DB_PATH = '/content/datasets_retry.duckdb'
RETRY_STATE_FILE = os.path.join(OUTPUT_DIR, "processed_stocks_retry.txt")

BATCH_SIZE = 20

print("🛡️ STARTING CONSOLIDATED ROBUST VERIFICATION AND RECOVERY (FIXED) 🛡️")
print("====================================================================")

# --- 1. Environment Setup ---
print("\n[1/6] Setting up Environment...")
try:
    print("   🔄 Force remounting Drive...")
    drive.flush_and_unmount()
except Exception as e:
    print(f"   ℹ️ Unmount skipped: {e}")
drive.mount('/content/drive', force_remount=True)

if not os.path.exists(PROJECT_PATH):
    print("   📥 Cloning repository...")
    try:
        token = userdata.get('GITHUB_TOKEN')
        subprocess.run(f'git clone https://{token}@github.com/gblue1223/stock-bot2.git {PROJECT_PATH}', shell=True, check=True)
    except:
        subprocess.run(f'git clone https://github.com/gblue1223/stock-bot2.git {PROJECT_PATH}', shell=True, check=True)

if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)

if not os.path.exists(DB_PATH):
    print("   💾 Copying database to local...")
    if os.path.exists(DRIVE_DB_PATH):
        shutil.copy(DRIVE_DB_PATH, DB_PATH)
        print("   ✅ DB copied.")
    else:
        sys.exit(f"❌ Drive DB not found at {DRIVE_DB_PATH}")
else:
    print("   ✅ Local DB exists.")

# --- Helper: Robust Copy ---
def robust_copy(src, dst, retries=3, delay=2):
    for i in range(retries):
        try:
            shutil.copy(src, dst)
            return True
        except OSError as e:
            if i < retries - 1:
                time.sleep(delay)
                continue
            print(f"      ⚠️ Failed to copy {os.path.basename(src)} after {retries} attempts: {e}")
            return False
        except Exception as e:
            print(f"      ⚠️ Unexpected error copying {os.path.basename(src)}: {e}")
            return False
    return False

# --- 2. Load Ground Truth ---
print("\n[2/6] Loading Ground Truth...")
db_stocks = set()
target_col = '종목코드'
try:
    con = duckdb.connect(DB_PATH)
    try:
        tables = [x[0] for x in con.execute("SHOW TABLES").fetchall()]
        table_name = 'datasets' if 'datasets' in tables else tables[0]
        cols = [x[0] for x in con.execute(f"DESCRIBE {table_name}").fetchall()]
        if 'code' in cols: target_col = 'code'
        elif 'Code' in cols: target_col = 'Code'

        db_stocks = set(str(x[0]) for x in con.execute(f"SELECT DISTINCT {target_col} FROM {table_name}").fetchall())
        print(f"   ✅ DB contains {len(db_stocks)} stocks.")
    except Exception as e:
        print(f"   ⚠️ Error reading DB: {e}")
    con.close()
except Exception as e:
    sys.exit(f"❌ DB Connection Error: {e}")

# --- 3. Chunked Verification with Resume ---
print("\n[3/6] Verifying Parquet Files (Resume Mode)...")
verified_files = set()
verified_stocks = set()

# Load state
if os.path.exists(VERIFICATION_STATE_FILE):
    print(f"   📖 Loading state from {VERIFICATION_STATE_FILE}...")
    try:
        with open(VERIFICATION_STATE_FILE, "r") as f:
            for line in f:
                parts = line.strip().split(",")
                if len(parts) >= 2:
                    verified_files.add(parts[0])
                    verified_stocks.add(parts[1])
        print(f"   ✅ Resuming: {len(verified_files)} files previously verified.")
    except: pass

if not os.path.exists(LOCAL_TEMP_VERIFY):
    os.makedirs(LOCAL_TEMP_VERIFY)

all_parquet = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.parquet")))
files_to_process = [f for f in all_parquet if os.path.basename(f) not in verified_files]
print(f"   ℹ️ Files remaining to verify: {len(files_to_process)}")

def verify_batch(batch_files):
    results = []
    temp_paths = []
    # Copy
    for src in batch_files:
        dst = os.path.join(LOCAL_TEMP_VERIFY, os.path.basename(src))
        if robust_copy(src, dst):
            temp_paths.append(dst)

    # Verify
    if temp_paths:
        try:
            c = duckdb.connect()
            for path in temp_paths:
                fname = os.path.basename(path)
                try:
                    res_rows = []
                    # Try finding the code column
                    try:
                        # FIX: fetchall() instead of fetchone() to handle multiple stocks per file
                        res_rows = c.execute(f"SELECT DISTINCT {target_col} FROM read_parquet('{path}')").fetchall()
                    except:
                        for col in ['code', 'Code', '종목코드', 'symbol']:
                            try:
                                res_rows = c.execute(f"SELECT DISTINCT {col} FROM read_parquet('{path}')").fetchall()
                                if res_rows: break
                            except: continue

                    if res_rows:
                        for row in res_rows:
                            results.append((fname, str(row[0])))
                except: pass
            c.close()
        except Exception as e:
            print(f"      ⚠️ DuckDB error during batch verification: {e}")

    # Cleanup temp
    for path in temp_paths:
        try: os.remove(path)
        except: pass

    return results

# Process Batches
if files_to_process:
    num_batches = math.ceil(len(files_to_process) / BATCH_SIZE)
    with open(VERIFICATION_STATE_FILE, "a") as state_f:
        for i in tqdm(range(num_batches), desc="Verifying"):
            batch = files_to_process[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
            batch_results = verify_batch(batch)

            for fname, code in batch_results:
                # A file might contain multiple stocks, add them all
                verified_stocks.add(code)
                # Mark file as verified (even if it maps to multiple stocks, we just need to know we processed it)
                if fname not in verified_files:
                    verified_files.add(fname)
                    # Log one line per stock-file pair or just ensure file is marked
                    # state file format: filename,code
                    state_f.write(f"{fname},{code}\n")
            state_f.flush()
else:
    print("   ✅ All files verified.")

# --- 4. Identify Missing Stocks ---
print("\n[4/6] Checking Missing Stocks...")
missing_stocks = sorted(list(db_stocks - verified_stocks))
print(f"   📊 Found {len(verified_stocks)} / {len(db_stocks)} stocks.")
print(f"   📉 Missing: {len(missing_stocks)}")
if 0 < len(missing_stocks) < 20:
    print(f"      Examples: {missing_stocks}")

# --- 5. Recovery Execution ---
if missing_stocks:
    print(f"\n[5/6] Recovery - Generating {len(missing_stocks)} missing embeddings...")

    # Create Retry DB
    if os.path.exists(RETRY_DB_PATH): os.remove(RETRY_DB_PATH)
    con = duckdb.connect(RETRY_DB_PATH)
    con.execute(f"ATTACH '{DB_PATH}' AS source")

    where_clause = f"= '{missing_stocks[0]}'" if len(missing_stocks) == 1 else f"IN {tuple(missing_stocks)}"
    con.execute(f"CREATE TABLE datasets AS SELECT * FROM source.datasets WHERE {target_col} {where_clause}")
    con.close()
    print("   ✅ Retry DB created.")

    # Clear Retry State
    with open(RETRY_STATE_FILE, 'w') as f: pass

    # Run Script
    if os.path.exists(PROJECT_PATH):
        os.chdir(PROJECT_PATH)
        os.environ['MAX_TEMP_SIZE'] = '50'

        cmd = f'''python scripts/data/generate_embeddings_colab.py \
          --db_path "{RETRY_DB_PATH}" \
          --model_path "{MODEL_PATH}" \
          --output_dir "{OUTPUT_DIR}" \
          --state_file "{RETRY_STATE_FILE}" \
          --table_name "datasets" \
          --seq_len 120 \
          --batch_size 4096 \
          --device cuda'''

        get_ipython().system(cmd)
        print("   ✅ Recovery script finished.")

        # Wait for Drive sync
        time.sleep(5)

        # Quick scan of new files to update verified_stocks for final report
        print("   🔎 Scanning newly generated files...")
        all_files_now = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.parquet")))
        new_files = [f for f in all_files_now if os.path.basename(f) not in verified_files]
        if new_files:
            new_results = verify_batch(new_files)
            for fname, code in new_results:
                 verified_stocks.add(code)
                 verified_files.add(fname)
else:
    print("\n[5/6] Recovery - No missing stocks.")

# --- 6. Finalize and Cleanup ---
print("\n[6/6] Finalizing...")

# Final Report
final_missing = sorted(list(db_stocks - verified_stocks))
print("\n📊 FINAL REPORT")
print(f"   ✅ Total Verified: {len(verified_stocks)}")
print(f"   📉 Still Missing: {len(final_missing)}")
if not final_missing:
    print("   🎉 SUCCESS! All stocks are present and verified.")
else:
    print(f"   ❌ Alert: {len(final_missing)} stocks are still missing.")

# Cleanup
print("\n🧹 Cleaning up...")
for p in [RETRY_DB_PATH, RETRY_DB_PATH + '.wal', LOCAL_TEMP_VERIFY, VERIFICATION_STATE_FILE]:
    if os.path.exists(p):
        try:
            if os.path.isdir(p): shutil.rmtree(p)
            else: os.remove(p)
            print(f"   🗑️ Removed {os.path.basename(p)}")
        except: pass

print("\n✨ Done!")